In [ ]:
# Step 1: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Step 2: Load Dataset
# Generating a realistic house price dataset
np.random.seed(42)
n = 500

area = np.random.randint(500, 5000, size=n)
bedrooms = np.random.randint(1, 6, size=n)
bathrooms = np.random.randint(1, 4, size=n)
stories = np.random.randint(1, 4, size=n)
parking = np.random.randint(0, 4, size=n)
mainroad = np.random.choice(['yes', 'no'], size=n)
basement = np.random.choice(['yes', 'no'], size=n)
location = np.random.choice(['urban', 'suburban', 'rural'], size=n)

# Price based on features + some noise
location_factor = np.where(location == 'urban', 1.4,
                  np.where(location == 'suburban', 1.1, 0.8))
mainroad_factor = np.where(mainroad == 'yes', 1.1, 1.0)
basement_factor = np.where(basement == 'yes', 1.05, 1.0)

price = (
    area * 150
    + bedrooms * 25000
    + bathrooms * 15000
    + stories * 20000
    + parking * 10000
) * location_factor * mainroad_factor * basement_factor
price = price + np.random.normal(0, 20000, size=n)
price = np.clip(price, 50000, None).astype(int)

df = pd.DataFrame({
    'area': area,
    'bedrooms': bedrooms,
    'bathrooms': bathrooms,
    'stories': stories,
    'parking': parking,
    'mainroad': mainroad,
    'basement': basement,
    'location': location,
    'price': price
})

In [ ]:
# Step 3: Display Dataset
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Step 4: Dataset Information
print("Dataset Info:")
df.info()

In [ ]:
# Step 5: Dataset Shape
print(f"Dataset Shape: {df.shape[0]} rows and {df.shape[1]} columns")

In [ ]:
# Step 6: Statistical Summary
print("Statistical Summary:")
df.describe()

In [ ]:
# Step 7: Exploratory Data Analysis (EDA)

# --- Missing Values ---
print("Missing Values per Column:")
print(df.isnull().sum())
print()

In [ ]:
# --- Duplicate Values ---
print(f"Number of Duplicate Rows: {df.duplicated().sum()}")

In [ ]:
# --- Price Distribution ---
plt.figure(figsize=(8, 5))
plt.hist(df['price'], bins=30, color='steelblue', edgecolor='black')
plt.title('Distribution of House Prices')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# --- Histograms of Numerical Features ---
numerical_cols = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    axes[i].hist(df[col], bins=20, color='cornflowerblue', edgecolor='black')
    axes[i].set_title(f'Distribution of {col.capitalize()}')
    axes[i].set_xlabel(col.capitalize())
    axes[i].set_ylabel('Frequency')

# Hide the unused subplot
axes[-1].set_visible(False)

plt.suptitle('Histograms of Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation Matrix ---
correlation = df[numerical_cols + ['price']].corr()

fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.matshow(correlation, cmap='coolwarm')
plt.colorbar(cax)

labels = numerical_cols + ['price']
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='left')
ax.set_yticklabels(labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{correlation.iloc[i, j]:.2f}", ha='center', va='center', fontsize=9)

plt.title('Correlation Matrix', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# --- Scatter Plot: Area vs Price ---
plt.figure(figsize=(8, 5))
plt.scatter(df['area'], df['price'], alpha=0.5, color='darkorange', edgecolors='black', linewidths=0.3)
plt.title('Area vs House Price')
plt.xlabel('Area (sq ft)')
plt.ylabel('Price')
plt.tight_layout()
plt.show()

In [ ]:
# Step 8: Data Cleaning & Preprocessing

# Handle missing values (none expected in this dataset, but good practice)
df.dropna(inplace=True)

# Encode categorical variables using Label Encoding
le = LabelEncoder()
categorical_cols = ['mainroad', 'basement', 'location']

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

print("Categorical variables encoded. Updated dataset head:")
df.head()

In [ ]:
# Step 9: Feature Engineering
# Creating a useful combined feature: rooms_per_story
df['rooms_per_story'] = (df['bedrooms'] + df['bathrooms']) / df['stories']

print("Feature 'rooms_per_story' added.")
df[['bedrooms', 'bathrooms', 'stories', 'rooms_per_story']].head()

In [ ]:
# Step 10: Feature Selection
X = df.drop(columns=['price'])
y = df['price']

print("Features selected:")
print(list(X.columns))
print(f"\nTarget: price")

In [ ]:
# Step 11: Train-Test Split (80:20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")

In [ ]:
# Step 12: Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
print("Linear Regression model trained.")

In [ ]:
# Step 13: Predict using Linear Regression
lr_predictions = lr_model.predict(X_test)

In [ ]:
# Step 14: Evaluate Linear Regression
lr_mae = mean_absolute_error(y_test, lr_predictions)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_predictions))
lr_r2 = r2_score(y_test, lr_predictions)

print("Linear Regression Results")
print(f"  MAE  : {lr_mae:,.2f}")
print(f"  RMSE : {lr_rmse:,.2f}")
print(f"  R²   : {lr_r2:.4f}")

In [ ]:
# Step 15: Train Decision Tree Regressor
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)
print("Decision Tree model trained.")

In [ ]:
# Step 16: Predict using Decision Tree
dt_predictions = dt_model.predict(X_test)

In [ ]:
# Step 17: Evaluate Decision Tree
dt_mae = mean_absolute_error(y_test, dt_predictions)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_predictions))
dt_r2 = r2_score(y_test, dt_predictions)

print("Decision Tree Results")
print(f"  MAE  : {dt_mae:,.2f}")
print(f"  RMSE : {dt_rmse:,.2f}")
print(f"  R²   : {dt_r2:.4f}")

In [ ]:
# Step 18: Train Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
print("Random Forest model trained.")

In [ ]:
# Step 19: Predict using Random Forest
rf_predictions = rf_model.predict(X_test)

In [ ]:
# Step 20: Evaluate Random Forest
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest Results")
print(f"  MAE  : {rf_mae:,.2f}")
print(f"  RMSE : {rf_rmse:,.2f}")
print(f"  R²   : {rf_r2:.4f}")

In [ ]:
# Step 21: Compare All Three Models
model_comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest'],
    'MAE': [lr_mae, dt_mae, rf_mae],
    'RMSE': [lr_rmse, dt_rmse, rf_rmse],
    'R²': [lr_r2, dt_r2, rf_r2]
})

model_comparison['MAE'] = model_comparison['MAE'].map('{:,.2f}'.format)
model_comparison['RMSE'] = model_comparison['RMSE'].map('{:,.2f}'.format)
model_comparison['R²'] = model_comparison['R²'].map('{:.4f}'.format)

print("Model Comparison Table:")
print(model_comparison.to_string(index=False))

In [ ]:
# Step 22: Feature Importance from Random Forest
feature_names = X.columns.tolist()
importances = rf_model.feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("Feature Importance (Random Forest):")
print(importance_df.to_string(index=False))

In [ ]:
# Step 23: Plot Feature Importance
plt.figure(figsize=(9, 5))
plt.barh(importance_df['Feature'][::-1], importance_df['Importance'][::-1], color='mediumseagreen', edgecolor='black')
plt.title('Feature Importance - Random Forest')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# Step 24: Print the Best Performing Model based on R²
scores = {
    'Linear Regression': lr_r2,
    'Decision Tree': dt_r2,
    'Random Forest': rf_r2
}

best_model = max(scores, key=scores.get)
best_r2 = scores[best_model]

print(f"Best Performing Model: {best_model}")
print(f"R² Score             : {best_r2:.4f}")

In [ ]:
# Step 25: Final Interpretation
top_feature = importance_df.iloc[0]['Feature']
top_importance = importance_df.iloc[0]['Importance']

print("Final Interpretation")
print("="*50)
print()
print(f"1. R² Score ({best_model}): {best_r2:.4f}")
if best_r2 >= 0.85:
    print("   The R² score is strong, meaning the model explains a large portion")
    print("   of the variance in house prices. The model is a reliable predictor.")
elif best_r2 >= 0.70:
    print("   The R² score is moderate. The model captures the general trend")
    print("   but may miss some complexity in the data.")
else:
    print("   The R² score is relatively low. The model needs improvement,")
    print("   possibly through better features or hyperparameter tuning.")
print()
print(f"2. RMSE ({best_model}):")
rmse_val = {'Linear Regression': lr_rmse, 'Decision Tree': dt_rmse, 'Random Forest': rf_rmse}[best_model]
print(f"   RMSE = {rmse_val:,.2f}")
print("   RMSE represents the average deviation of predictions from actual prices.")
print(f"   On average, the model's predictions are off by approximately ${rmse_val:,.0f}.")
print("   Lower RMSE indicates better model performance.")
print()
print(f"3. Most Influential Feature: '{top_feature}'")
print(f"   Importance Score: {top_importance:.4f}")
print(f"   '{top_feature}' has the greatest impact on predicting house prices,")
print("   meaning changes in this feature produce the largest change in predicted price.")